# ID 5841 Quantum Computing – Assignment 3
Deutsch–Jozsa and Grover Algorithms

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit import transpile
from qiskit.visualization import plot_histogram

from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit_aer.noise.errors import amplitude_damping_error, pauli_error

import numpy as np
import math

---
# Problem 1: Deutsch–Jozsa Algorithm
f(x₀, x₁, x₂) = x₀ ⊕ (x₁ ∧ x₂)

## 1(i) Classical Analysis

In [ ]:
def f_classical(x0, x1, x2):
    return x0 ^ (x1 & x2)

print("x0 x1 x2 | f")
print("-----------")
ones = 0
for x0 in [0, 1]:
    for x1 in [0, 1]:
        for x2 in [0, 1]:
            val = f_classical(x0, x1, x2)
            ones += val
            print(f" {x0}  {x1}  {x2} | {val}")
print(f"\nOnes: {ones}, Zeros: {8 - ones}")

**Observations:** The truth table has exactly 4 ones and 4 zeros, so f is **balanced**. 
Classically, a worst-case black-box evaluation requires **5 queries** (need more than half the outputs to confirm balance).

## 1(ii) DJ Oracle and Noiseless Simulation

In [ ]:
def deutsch_jozsa_oracle_f():
    """
    Oracle Uf for f(x0,x1,x2) = x0 XOR (x1 AND x2).
    q[0..2] = inputs, q[3] = output ancilla.
    Uf|x>|y> = |x>|y XOR f(x)>
    """
    qc_oracle = QuantumCircuit(4, name="U_f")
    # XOR contribution from x0: CNOT(x0 -> ancilla)
    qc_oracle.cx(0, 3)
    # XOR contribution from x1 AND x2: Toffoli(x1, x2 -> ancilla)
    qc_oracle.ccx(1, 2, 3)
    return qc_oracle

deutsch_jozsa_oracle_f().draw('mpl')

In [ ]:
# Full Deutsch-Jozsa circuit (noiseless)
dj_qc = QuantumCircuit(4, 3)

# Initialise |0001>: X on output qubit
dj_qc.x(3)

# Hadamard on all 4 qubits
dj_qc.h([0, 1, 2, 3])

# Append oracle
oracle = deutsch_jozsa_oracle_f()
dj_qc.append(oracle.to_gate(), [0, 1, 2, 3])

# Hadamard on input qubits
dj_qc.h([0, 1, 2])

# Measure input qubits
dj_qc.measure([0, 1, 2], [0, 1, 2])

print(dj_qc.draw())

noiseless_sim = AerSimulator()
dj_qc_t = transpile(dj_qc, noiseless_sim)
result = noiseless_sim.run(dj_qc_t, shots=1024).result()
counts = result.get_counts()
print("Counts:", counts)
plot_histogram(counts, title="DJ Noiseless")

**Observations:** The output never shows `000`, confirming f is **balanced** (a constant function would yield `000` with probability 1). A single quantum oracle query suffices.

## 1(iii) DJ with Noise

In [ ]:
# Noise model: bit-flip errors on single-qubit gates + amplitude damping on CX
noise_model = NoiseModel()

p_bitflip = 0.01
gamma_amp = 0.02

bitflip_err = pauli_error([('X', p_bitflip), ('I', 1 - p_bitflip)])
ampdamp_err = amplitude_damping_error(gamma_amp)

noise_model.add_all_qubit_quantum_error(bitflip_err, ['h', 'x'])
noise_model.add_all_qubit_quantum_error(ampdamp_err, ['cx'])

noisy_sim = AerSimulator(noise_model=noise_model)
dj_qc_noisy = transpile(dj_qc, noisy_sim)
result_noisy = noisy_sim.run(dj_qc_noisy, shots=1024).result()
counts_noisy = result_noisy.get_counts()
print("Noisy counts:", counts_noisy)

plot_histogram([counts, counts_noisy],
               title="DJ: Noiseless vs Noisy",
               legend=["Noiseless", "Noisy"])

**Observations:** Noise spreads probability mass to other bit strings, including `000`. Since `000` is the signature of a *constant* function, its nonzero noisy count could falsely suggest the function is constant. The higher the error rate, the less reliable the DJ algorithm becomes at distinguishing constant vs balanced.

---
# Problem 2: Grover Search for 3-SAT
f(x₁,x₂,x₃) = (x₁∨x₂∨x₃) ∧ (¬x₁∨x₂∨x₃) ∧ (¬x₁∨¬x₂∨x₃) ∧ (¬x₁∨¬x₂∨¬x₃) ∧ (x₁∨x₂∨¬x₃) ∧ (¬x₁∨x₂∨¬x₃)

## 2(i) Truth Table

In [ ]:
def f_3sat(x1, x2, x3):
    c1 = x1 | x2 | x3
    c2 = (1 - x1) | x2 | x3
    c3 = (1 - x1) | (1 - x2) | x3
    c4 = (1 - x1) | (1 - x2) | (1 - x3)
    c5 = x1 | x2 | (1 - x3)
    c6 = (1 - x1) | x2 | (1 - x3)
    return int(c1 & c2 & c3 & c4 & c5 & c6)

print("x1 x2 x3 | f")
print("-----------")
solutions = []
for x1 in [0, 1]:
    for x2 in [0, 1]:
        for x3 in [0, 1]:
            val = f_3sat(x1, x2, x3)
            print(f" {x1}  {x2}  {x3} | {val}")
            if val == 1:
                solutions.append((x1, x2, x3))
print(f"\nSatisfying assignments: {solutions}")

**Observations:** Exactly two satisfying assignments: **(x₁,x₂,x₃) = (0,1,0) and (0,1,1)**. All other 6 inputs evaluate to 0.

## 2(ii) Grover Oracle

In [ ]:
def grover_oracle_3sat():
    """
    Phase-flip oracle for the 3-SAT problem.
    Marks states (x1=0,x2=1,x3=0) and (x1=0,x2=1,x3=1).
    Qiskit ordering: q[0]=x1, q[1]=x2, q[2]=x3.
    Measured strings: '010' (x1=0,x2=1,x3=0) and '110' (x1=0,x2=1,x3=1)
    in Qiskit little-endian readout c[2]c[1]c[0].
    Strategy: X-gates map each target state to |111>, CCZ flips phase, X-gates undo.
    """
    qc = QuantumCircuit(3, name="GroverOracle")

    # --- Mark (x1=0, x2=1, x3=0): state |q2=0, q1=1, q0=0> ---
    # Flip q0 and q2 (the zero bits) to reach |111>
    qc.x([0, 2])
    # CCZ via H + CCX + H
    qc.h(2)
    qc.ccx(0, 1, 2)
    qc.h(2)
    # Undo
    qc.x([0, 2])

    # --- Mark (x1=0, x2=1, x3=1): state |q2=1, q1=1, q0=0> ---
    # Flip q0 only (the zero bit) to reach |111>
    qc.x([0])
    # CCZ via H + CCX + H
    qc.h(2)
    qc.ccx(0, 1, 2)
    qc.h(2)
    # Undo
    qc.x([0])

    return qc

grover_oracle_3sat().draw('mpl')

## 2(iii) Grover Search — Noiseless

In [ ]:
def diffuser_3qubits():
    """3-qubit Grover diffuser (inversion about the mean)."""
    qc = QuantumCircuit(3, name="Diffuser")
    qc.h([0, 1, 2])
    qc.x([0, 1, 2])
    qc.h(2)
    qc.ccx(0, 1, 2)
    qc.h(2)
    qc.x([0, 1, 2])
    qc.h([0, 1, 2])
    return qc

In [ ]:
# Optimal number of Grover iterations: r = floor(pi/4 * sqrt(N/M))
N, M = 8, 2
r = math.floor(math.pi / 4 * math.sqrt(N / M))
print(f"N={N}, M={M} -> optimal iterations r = {r}")

# Build Grover circuit
grover_qc = QuantumCircuit(3, 3)
grover_qc.h([0, 1, 2])  # Equal superposition

oracle = grover_oracle_3sat()
diff = diffuser_3qubits()

for _ in range(r):
    grover_qc.append(oracle.to_gate(), [0, 1, 2])
    grover_qc.append(diff.to_gate(), [0, 1, 2])

grover_qc.measure([0, 1, 2], [0, 1, 2])

print(grover_qc.draw())

grover_sim = AerSimulator()
grover_qc_t = transpile(grover_qc, grover_sim)
result_grover = grover_sim.run(grover_qc_t, shots=1024).result()
counts_grover = result_grover.get_counts()
print("Counts:", counts_grover)
plot_histogram(counts_grover, title="Grover 3-SAT (Noiseless)")

**Observations:** With N=8 and M=2 solutions, the optimal number of Grover iterations is **r=1**. The histogram peaks sharply at the two satisfying assignments (`010` and `110` in Qiskit's little-endian bit ordering, corresponding to (x₁=0,x₂=1,x₃=0) and (x₁=0,x₂=1,x₃=1)). The oracle is queried only once.

## 2(iv) Grover Search — Noisy

In [ ]:
# Reuse noise_model defined in Problem 1
noisy_grover_sim = AerSimulator(noise_model=noise_model)
grover_qc_noisy = transpile(grover_qc, noisy_grover_sim)
result_grover_noisy = noisy_grover_sim.run(grover_qc_noisy, shots=1024).result()
counts_grover_noisy = result_grover_noisy.get_counts()
print("Noisy counts:", counts_grover_noisy)

plot_histogram(
    [counts_grover, counts_grover_noisy],
    title="Grover 3-SAT: Noiseless vs Noisy",
    legend=["Noiseless", "Noisy"]
)

**Observations:** Noise reduces the peak probabilities at the two satisfying states and spreads counts to other bit strings. Since r=1 involves relatively few gates, the degradation is moderate — the two solutions remain the most probable outcomes but with lower confidence. Increasing noise rates would further flatten the distribution, eventually making it indistinguishable from a uniform distribution.